# Matrix Plots with Seaborn

Matrix plots display data that is naturally organized in a 2D grid — correlation matrices, pivot tables, confusion matrices, and more. Seaborn provides two key functions for this:

- `sns.heatmap()` — color-coded matrix visualization
- `sns.clustermap()` — heatmap with hierarchical clustering

This notebook covers:
- Building and visualizing correlation matrices
- Customizing heatmaps (annotations, colormaps, masking)
- Creating heatmaps from pivot tables
- Hierarchical clustering with `clustermap()`
- Practical correlation analysis workflow

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
sns.set_theme(style='white')

In [ ]:
tips = sns.load_dataset('tips')
flights = sns.load_dataset('flights')

print(f'tips: {tips.shape}')
print(f'flights: {flights.shape}')

## Basic Heatmap: `sns.heatmap()`

`heatmap()` takes a 2D array-like or DataFrame and maps values to a color gradient.

In [ ]:
# Simple heatmap from a random matrix
np.random.seed(42)
data = np.random.rand(5, 7)

plt.figure(figsize=(8, 4))
sns.heatmap(data)
plt.title('Heatmap of Random Data')
plt.show()

## Correlation Matrix Heatmap

One of the most common uses: compute the Pearson correlation between numeric columns and display it as a heatmap.

In [ ]:
# Compute correlation matrix for the tips dataset
tips_numeric = tips.select_dtypes(include='number')
corr = tips_numeric.corr()
corr

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Tips Dataset — Correlation Matrix')
plt.show()

### Customizing Heatmap Appearance

Key parameters:
- `annot` — display numeric values in each cell
- `fmt` — format string for annotations (e.g., `'.2f'`, `'d'`)
- `cmap` — colormap name
- `center` — value at the center of the colormap (useful for diverging maps)
- `linewidths` — width of lines separating cells
- `linecolor` — color of separating lines
- `vmin`, `vmax` — bounds for the color scale
- `square` — force square cells

In [ ]:
plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    annot=True, fmt='.2f',
    cmap='RdBu_r', center=0,
    linewidths=1, linecolor='white',
    square=True,
    vmin=-1, vmax=1,
    cbar_kws={'label': 'Correlation', 'shrink': 0.8},
)
plt.title('Styled Correlation Heatmap')
plt.show()

### Masking the Upper Triangle

Since a correlation matrix is symmetric, showing only the lower triangle avoids redundancy.

In [ ]:
# Create a boolean mask for the upper triangle
mask = np.triu(np.ones_like(corr, dtype=bool))

plt.figure(figsize=(7, 5))
sns.heatmap(
    corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=0.5,
    square=True,
)
plt.title('Lower Triangle Correlation Matrix')
plt.show()

## Heatmaps from Pivot Tables

The flights dataset is in long format. We can pivot it into a month-by-year matrix, then visualize with a heatmap.

In [ ]:
# Pivot the flights data
flights_pivot = flights.pivot_table(
    index='month', columns='year', values='passengers',
)
flights_pivot.head()

In [ ]:
plt.figure(figsize=(12, 7))
sns.heatmap(
    flights_pivot,
    annot=True, fmt='d',
    cmap='YlOrRd',
    linewidths=0.5,
)
plt.title('Airline Passengers per Month (1949-1960)')
plt.ylabel('Month')
plt.xlabel('Year')
plt.show()

In [ ]:
# Without annotations for a cleaner look on dense matrices
plt.figure(figsize=(12, 7))
sns.heatmap(
    flights_pivot,
    cmap='viridis',
    linewidths=0.3, linecolor='gray',
    cbar_kws={'label': 'Passengers (thousands)'},
)
plt.title('Passengers Heatmap — Viridis Colormap')
plt.show()

### Different Colormaps

Choosing the right colormap matters:
- **Sequential** (`viridis`, `YlOrRd`, `Blues`) — data goes from low to high
- **Diverging** (`coolwarm`, `RdBu_r`, `PiYG`) — data has a meaningful center (e.g., 0 for correlation)
- **Qualitative** — avoid for heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

cmaps = ['Blues', 'magma', 'YlGnBu']
for ax, cmap in zip(axes, cmaps):
    sns.heatmap(flights_pivot, cmap=cmap, ax=ax)
    ax.set_title(f'cmap = {cmap}')
    ax.set_xlabel('')
    ax.set_ylabel('')

plt.tight_layout()
plt.show()

## Cluster Maps: `sns.clustermap()`

`clustermap()` performs **hierarchical clustering** on both rows and columns and reorders them so that similar items are adjacent. It draws dendrograms on the margins to show the clustering structure.

In [ ]:
# Cluster map of the flights pivot table
g = sns.clustermap(
    flights_pivot,
    cmap='YlOrRd',
    annot=True, fmt='d',
    figsize=(12, 8),
    linewidths=0.5,
)
g.fig.suptitle('Clustered Heatmap of Airline Passengers', y=1.02)
plt.show()

In [ ]:
# Cluster only rows (keep year order)
g = sns.clustermap(
    flights_pivot,
    cmap='viridis',
    col_cluster=False,  # keep columns in original order
    figsize=(12, 8),
    linewidths=0.3,
)
g.fig.suptitle('Row-Clustered Passengers (Years in Order)', y=1.02)
plt.show()

In [ ]:
# Standardize rows to compare seasonal patterns regardless of magnitude
g = sns.clustermap(
    flights_pivot,
    standard_scale=0,  # 0 = standardize rows
    cmap='RdBu_r',
    figsize=(12, 8),
    linewidths=0.3,
)
g.fig.suptitle('Row-Standardized Cluster Map', y=1.02)
plt.show()

## Practical Workflow: Correlation Analysis

Bringing it all together with a more realistic dataset — the penguins data.

In [ ]:
penguins = sns.load_dataset('penguins').dropna()
pen_numeric = penguins.select_dtypes(include='number')
pen_corr = pen_numeric.corr()

# Masked lower-triangle correlation heatmap
mask = np.triu(np.ones_like(pen_corr, dtype=bool))

plt.figure(figsize=(8, 6))
sns.heatmap(
    pen_corr,
    mask=mask,
    annot=True, fmt='.2f',
    cmap='coolwarm', center=0,
    linewidths=1, square=True,
    vmin=-1, vmax=1,
)
plt.title('Penguin Measurements — Correlation Matrix')
plt.show()

In [ ]:
# Per-species correlation heatmaps
species_list = penguins['species'].unique()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, sp in zip(axes, species_list):
    sp_corr = penguins[penguins['species'] == sp].select_dtypes(include='number').corr()
    mask = np.triu(np.ones_like(sp_corr, dtype=bool))
    sns.heatmap(
        sp_corr, mask=mask,
        annot=True, fmt='.2f',
        cmap='coolwarm', center=0,
        vmin=-1, vmax=1, ax=ax,
        square=True,
    )
    ax.set_title(f'{sp}')

plt.suptitle('Correlation Matrices by Penguin Species', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Pivot table heatmap: mean body mass by species and island
mass_pivot = penguins.pivot_table(
    index='species', columns='island',
    values='body_mass_g', aggfunc='mean',
)

plt.figure(figsize=(7, 4))
sns.heatmap(
    mass_pivot,
    annot=True, fmt='.0f',
    cmap='YlGnBu',
    linewidths=1, linecolor='white',
)
plt.title('Mean Body Mass (g) by Species and Island')
plt.show()

## Key Takeaways

| Function | Purpose |
|---|---|
| `heatmap()` | Color-coded 2D matrix visualization |
| `clustermap()` | Heatmap + hierarchical clustering with dendrograms |

- Always use `annot=True` with `fmt` to display values when the matrix is small enough to read.
- Use **diverging** colormaps (`coolwarm`, `RdBu_r`) with `center=0` for correlation matrices.
- Use **sequential** colormaps (`viridis`, `YlOrRd`) for counts or magnitudes.
- Mask the upper triangle of correlation matrices with `np.triu()` to avoid redundancy.
- `clustermap()` is powerful for discovering natural groupings — use `standard_scale` to normalize before clustering when variables have different scales.
- Build heatmap-ready DataFrames with `pd.pivot_table()` or `df.corr()`.